# Visual-Doc Assistant — Phase 3
## Gemini Answer Generation & Streamlit UI

**Objective:** Take a user's natural-language query, retrieve the most relevant page(s) from the ChromaDB index built in Phase 2, pass those page images + the query to Gemini for multimodal answer generation, and expose the whole pipeline through a Streamlit UI — completing the end-to-end Visual-Doc Assistant.

---
### Phase Context
```
Phase 1  →  PDF → Images + ChromaDB initialised (0 embeddings)
Phase 2  →  Images → ColPali → 128-dim Embeddings → ChromaDB
           +Query → ColPali → 128-dim Query Vector → Sanity Check
Phase 3  →  Query → ColPali embed_query() → ChromaDB retrieval → Top-K Pages
           → Gemini multimodal → Answer → Streamlit UI              THIS NOTEBOOK
```

> **GPU required:** `Runtime → Change runtime type → T4 GPU`. ColPali still needs to embed the *live* query at question-time using the same model from Phase 2, so this notebook reloads it.

> **Heads-up on the Phase 2 notebook, before you run Step 1:** `Phase2_Implementation.ipynb`'s install cell pins `colpali-engine==0.3.7`, which isn't published on PyPI, and its Drive wheel cache has no logic to detect stale versions. Phase 2's results are all verified end-to-end, so this was evidently patched by hand in the live session at the time, but the fix never made it back into the saved notebook. Step 1 below installs `colpali-engine==0.3.17` instead (the current release). Worth back-porting this same install cell into Phase 2 next time you touch it, so a cold restart of that notebook doesn't break.
>
> **Second correction, if you hit a `NotImplementedError: Cannot copy out of meta tensor` on Step 2:** an earlier version of this Step 1 cell installed `peft` and `torchao` with no upper version bound. `colpali-engine`'s own published metadata pins `peft>=0.18.0,<0.20.0` and `transformers>=5.3.0,<6.0.0` specifically — its LoRA adapter key-remapping fix for this exact checkpoint's `custom_text_proj` layer (landed in colpali-engine v0.3.14) was built and tested against that range, and an unconstrained `peft` install can resolve outside it, breaking that remapping and leaving part of the model uninitialized on the "meta" device. Step 1 now pins `peft`/`transformers` to colpali-engine's own declared ranges instead of leaving them open, and `torchao` is dropped entirely — it isn't actually a colpali-engine dependency and nothing here imports it. **If you already hit this error, a plain "Restart runtime" won't fix it** — Colab keeps pip-installed packages across a kernel restart within the same VM. Use **Runtime → Disconnect and delete runtime**, reconnect, and re-run from Step 1 below.


## Step 1 — Mount Drive & Reinstall Dependencies

Same Drive-cached-wheel folder as Phase 2 (`Visual-Doc/deps/`, so packages persist across ephemeral Colab sessions), but installed with `--find-links` rather than `--no-index`: the cache is used when it's valid, but pip can still fall back to PyPI to correct anything stale or missing, instead of silently installing whatever happens to already be sitting in the cache or the environment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, os
for pattern in ['pillow*', 'torchao*']:
    for f in glob.glob(f'./deps/{pattern}'):
        os.remove(f)
        print('removed', f)

In [ ]:
import sys, os, subprocess

WHEELS_DIR = './deps'
os.makedirs(WHEELS_DIR, exist_ok=True)

# colpali-engine==0.3.7 (as pinned in Phase2_Implementation.ipynb) is not on PyPI — using
# the current release instead. Its own published metadata pins peft>=0.18.0,<0.20.0 and
# transformers>=5.3.0,<6.0.0 (confirmed against PyPI + the colpali-engine v0.3.14 changelog,
# which is where its LoRA adapter key-remapping fix for this checkpoint's custom_text_proj
# layer landed) — declaring them explicitly here, in the SAME install command as
# colpali-engine, so pip resolves all of them together instead of picking an incompatible
# version for one of them separately. torchao is dropped: it's not an actual colpali-engine
# dependency and nothing in this notebook imports it.
CORE = [
    'colpali-engine==0.3.17',
    'peft>=0.18.0,<0.20.0',
    'transformers>=5.3.0,<6.0.0',
    'chromadb',
    'Pillow<12.0.0',
    'torchao>=0.16.0',
]

def pip_run(cmd, label):
    """Run a pip command, print a clean one-line result."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    our = ['colpali', 'chromadb', 'pillow', 'peft', 'transformers']
    errs = [l for l in result.stderr.splitlines()
            if 'ERROR' in l and any(p in l.lower() for p in our)]
    if errs:
        print(f'  {label}:')
        for e in errs: print(f'   {e}')
    else:
        print(f' {label}')

# Populate/refresh the Drive wheel cache (cheap no-op on repeat runs once it's warm)
pip_run(
    [sys.executable, '-m', 'pip', 'download', '--dest', WHEELS_DIR, '--quiet'] + CORE,
    f'Wheels cached to {WHEELS_DIR}'
)

# Install with --find-links (prefer the Drive cache) but deliberately WITHOUT --no-index,
# and with --upgrade: pip can still reach PyPI to correct anything the cache is missing or
# that's the wrong version, rather than silently keeping a stale/incompatible package in
# place. --no-index (used in the first attempt at this cell) is what let an unconstrained
# peft install go unnoticed instead of being corrected here.
pip_run(
    [sys.executable, '-m', 'pip', 'install',
     '--find-links', WHEELS_DIR, '--upgrade', '--quiet', '--no-warn-conflicts'] + CORE,
    'Core packages installed'
)

print()
try:
    import colpali_engine, chromadb, transformers, peft
    print(f' colpali_engine : {getattr(colpali_engine, "__version__", "n/a")}')
    print(f' transformers   : {transformers.__version__}')
    print(f' peft           : {peft.__version__}')
    print(f' chromadb       : {chromadb.__version__}')
except ImportError as e:
    print(f' Import failed: {e}')

## Step 2 — Reload ColPali Vision-Language Model

Identical to Phase 2, Step 2. Needed here because retrieval at query-time requires encoding the user's live question into the same 128-dim space the stored page embeddings live in.

In [ ]:
import torch
from PIL import Image
from IPython.display import display, Image as IPImage
from colpali_engine.models import ColPali, ColPaliProcessor

MODEL_NAME = 'vidore/colpali-v1.2'
EMBED_DIM  = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device   : {DEVICE}')
print(f'Embed dim: {EMBED_DIM}')
if DEVICE == 'cpu':
    print('  No GPU detected — switch to T4 GPU runtime before continuing.')

print(f'\nLoading {MODEL_NAME} ...')
model = ColPali.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()

processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

print(f'\n ColPali loaded on {DEVICE} — ready to embed live queries.')

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

adapter_path = hf_hub_download('vidore/colpali-v1.2', filename='adapter_model.safetensors')
adapter_state_dict = load_file(adapter_path)

target_dtype  = next(model.parameters()).dtype
target_device = next(model.parameters()).device

def remap_key(k):
    if k.startswith('base_model.model.'):
        k = k[len('base_model.model.'):]
    if k.endswith('lora_A.weight') or k.endswith('lora_B.weight'):
        k = k[:-len('.weight')] + '.default.weight'
    return k

remapped = {
    remap_key(k): v.to(dtype=target_dtype, device=target_device)
    for k, v in adapter_state_dict.items()
}

model.load_state_dict(remapped, strict=False)

# lora_B is zero-initialized by default — a nonzero norm here confirms the
# real trained weights made it in, not just a freshly-initialized adapter.
b_norm = model.custom_text_proj.lora_B.default.weight.norm().item()
print(f'custom_text_proj lora_B weight norm: {b_norm:.4f}')
print('Loaded correctly.' if b_norm > 0 else 'Still zero — not loaded.')

## Step 3 — Reconnect to Phase 2's ChromaDB Collection

No re-embedding here — Phase 2 already populated all 26 pages. This just reconnects to that existing collection on Drive and verifies it's intact.

In [ ]:
import os
import numpy as np
import chromadb

IMAGE_FOLDER = './processed_images'
DB_PATH      = './chroma_db_storage'

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection(
    name='visual_doc_collection',
    metadata={'hnsw:space': 'cosine'}
)

print(f'Collection "{collection.name}" — {collection.count()} embeddings stored.')
assert collection.count() > 0, "No embeddings found — run Phase 2 first."
print('Phase 2 index verified. Ready for retrieval.')

## Step 4 — `embed_query()` (unchanged from Phase 2)

Reproduced here rather than imported, so this notebook is self-contained and runnable on its own in a fresh runtime. Kept byte-for-byte identical to Phase 2 so retrieval behaves exactly as already verified there.

In [ ]:
@torch.no_grad()
def embed_query(query_text: str) -> np.ndarray:
    """
    Generate a 128-dim embedding for a text query using ColPali's text encoder.
    Same model and projection layer as Phase 2's embed_page_image — query and
    image embeddings live in the same semantic space.
    """
    inputs           = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)                          # (1, N_tokens, 128)
    pooled           = token_embeddings.mean(dim=1).squeeze(0)  # (128,)
    return pooled.cpu().float().numpy()

## Step 5 — Retrieval Function

Adapted from Phase 2's `retrieval_sanity_check` — same embed → cosine search logic, but returns structured results instead of just printing/displaying, so Gemini prompt construction and the Streamlit UI can consume them directly.

In [ ]:
def retrieve_top_k_pages(query_text: str, k: int = 3) -> list:
    """
    Embed the query with ColPali and return the top-k most similar pages
    from the Phase 2 ChromaDB index.
    """
    query_embedding = embed_query(query_text)

    results = collection.query(
        query_embeddings = [query_embedding.tolist()],
        n_results        = k,
        include          = ['metadatas', 'distances'],
    )

    pages = []
    for meta, dist in zip(results['metadatas'][0], results['distances'][0]):
        pages.append({
            'page_number': meta['page_number'],
            'image_path':  meta['image_path'],
            'similarity':  1 - dist,   # ChromaDB returns cosine distance
        })
    return pages

## Step 6 — Install & Configure the Gemini API

Uses the `google-genai` SDK and its Interactions API (the current recommended way to call Gemini — the older `generateContent`-only style and the deprecated `google-generativeai` package are both superseded by this). Get a free API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).

In [ ]:
!pip install -q -U google-genai

import os
from getpass import getpass
from google import genai

if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = getpass('Enter your Gemini API key (from https://aistudio.google.com/apikey): ')

# Named gemini_client (not "client") — "client" is already the ChromaDB PersistentClient from Step 3
gemini_client = genai.Client()
GEMINI_MODEL  = 'gemini-3.6-flash'   # strong multimodal quality/latency balance
                                     # swap to 'gemini-3.5-flash-lite' for a faster/cheaper option

print(f' Gemini client ready — using {GEMINI_MODEL}')

## Step 7 — Multimodal Prompt Construction & Answer Generation

Retrieved page images are sent inline as base64 (well within the Interactions API's 20MB inline-request limit for a handful of 300-DPI page PNGs — see Phase 1's ~200KB–1MB-per-page estimate). A system instruction grounds Gemini in *only* the retrieved pages, so it doesn't fall back on outside knowledge.

In [ ]:
import base64

SYSTEM_INSTRUCTION = (
    "You are the Visual-Doc Assistant, a technical documentation Q&A system. "
    "Answer the user's question using ONLY the page image(s) provided below — "
    "do not use outside knowledge. Reference page numbers when relevant. "
    "If the provided pages don't contain the answer, say so directly instead of guessing."
)

def build_multimodal_input(query_text: str, pages: list) -> list:
    """Builds the Gemini `input` list: query text first, then each retrieved page image inline."""
    content = [{'type': 'text', 'text': f'Question: {query_text}'}]
    for p in pages:
        with open(p['image_path'], 'rb') as f:
            image_bytes = f.read()
        content.append({
            'type':      'image',
            'data':      base64.b64encode(image_bytes).decode('utf-8'),
            'mime_type': 'image/png',
        })
    return content

def generate_answer(query_text: str, pages: list) -> str:
    """Send the query + retrieved page images to Gemini and return its text answer."""
    try:
        interaction = gemini_client.interactions.create(
            model              = GEMINI_MODEL,
            input              = build_multimodal_input(query_text, pages),
            system_instruction = SYSTEM_INSTRUCTION,
        )
        return interaction.output_text
    except Exception as e:
        return f" Gemini request failed: {e}"


## Step 8 — End-to-End QA Pipeline & Sanity Checks

Same two queries used in Phase 2's retrieval sanity check — now with a real generated answer attached, proving the full Phase 3 loop (embed → retrieve → generate → display) end to end.

In [ ]:
def ask(query_text: str, k: int = 3, show_pages: bool = True):
    """
    Full Phase 3 pipeline: embed the query, retrieve top-k pages from ChromaDB,
    send them + the query to Gemini, and print the answer alongside source pages.
    """
    print(f'Query: "{query_text}"\n')
    pages = retrieve_top_k_pages(query_text, k=k)

    print(f'Top-{k} retrieved pages:')
    for p in pages:
        print(f"  Page {p['page_number']:>2} — cosine similarity: {p['similarity']:.4f}")

    answer = generate_answer(query_text, pages)
    print('\nGemini answer:')
    print(answer)

    if show_pages:
        print('\nSource pages:')
        for p in pages:
            img = Image.open(p['image_path']).convert('RGB')
            img.thumbnail((400, 600))
            display(img)

    print()
    return answer, pages


# ── Sanity checks — same queries as Phase 2, now with real answer generation ──
ask('What is the system architecture and pipeline diagram?', k=3)
ask('What are the results and performance comparisons?', k=3)

## Step 9 — Streamlit UI

Writes a standalone `app.py`. It reloads its own copy of everything above (a Streamlit app runs as a separate process, so it can't import the cells above directly) but the logic is identical, wrapped in `@st.cache_resource` so the ColPali model and ChromaDB connection load once per session rather than on every query.

In [ ]:
%%writefile /content/app.py
"""
Visual-Doc Assistant — Phase 3 Streamlit UI
Run with: streamlit run /content/app.py   (see the next cell for launching it from Colab)
"""
import os
import base64

import streamlit as st
import torch
from PIL import Image
import chromadb
from colpali_engine.models import ColPali, ColPaliProcessor
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from google import genai

# ── Config (mirrors the notebook above) ─────────────────────────────────────
MODEL_NAME   = 'vidore/colpali-v1.2'
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_FOLDER = './processed_images'
DB_PATH      = './chroma_db_storage'
GEMINI_MODEL = 'gemini-3.6-flash'

SYSTEM_INSTRUCTION = (
    "You are the Visual-Doc Assistant, a technical documentation Q&A system. "
    "Answer the user's question using ONLY the page image(s) provided below — "
    "do not use outside knowledge. Reference page numbers when relevant. "
    "If the provided pages don't contain the answer, say so directly instead of guessing."
)

st.set_page_config(page_title="Visual-Doc Assistant", page_icon="📄", layout="wide")

# ── Cached resources: load once per session, not once per query ────────────
def _patch_custom_text_proj(model):
    """
    The vidore/colpali-v1.2 checkpoint stores its custom_text_proj LoRA weights
    in the old PEFT save format ("base_model.model.custom_text_proj.lora_A.weight"),
    but transformers' automatic adapter loading expects
    "custom_text_proj.lora_A.default.weight". That mismatch leaves this layer at
    its random (zero lora_B) init instead of the trained values, so we reload it
    manually with the correct key names.
    """
    adapter_path = hf_hub_download(MODEL_NAME, filename='adapter_model.safetensors')
    adapter_state_dict = load_file(adapter_path)

    target_dtype = next(model.parameters()).dtype
    target_device = next(model.parameters()).device

    def remap_key(k):
        if k.startswith('base_model.model.'):
            k = k[len('base_model.model.'):]
        if k.endswith('lora_A.weight') or k.endswith('lora_B.weight'):
            k = k[:-len('.weight')] + '.default.weight'
        return k

    remapped = {
        remap_key(k): v.to(dtype=target_dtype, device=target_device)
        for k, v in adapter_state_dict.items()
    }
    model.load_state_dict(remapped, strict=False)

    b_norm = model.custom_text_proj.lora_B.default.weight.norm().item()
    if b_norm == 0:
        raise RuntimeError(
            'custom_text_proj LoRA weights still zero after manual patch — '
            'retrieval quality would be degraded. Check the key remap logic.'
        )
    return model


@st.cache_resource(show_spinner="Loading ColPali model (first load only)...")
def load_colpali():
    model = ColPali.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map=DEVICE,
    ).eval()
    model = _patch_custom_text_proj(model)
    processor = ColPaliProcessor.from_pretrained(MODEL_NAME)
    return model, processor

@st.cache_resource(show_spinner="Connecting to ChromaDB...")
def load_collection():
    db_client = chromadb.PersistentClient(path=DB_PATH)
    return db_client.get_or_create_collection(
        name='visual_doc_collection', metadata={'hnsw:space': 'cosine'}
    )

model, processor = load_colpali()
collection = load_collection()

# ── Pipeline functions (identical logic to the Phase 3 notebook) ───────────
@torch.no_grad()
def embed_query(query_text):
    inputs = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)
    pooled = token_embeddings.mean(dim=1).squeeze(0)
    return pooled.cpu().float().numpy()

def retrieve_top_k_pages(query_text, k=3):
    query_embedding = embed_query(query_text)
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k,
        include=['metadatas', 'distances'],
    )
    return [
        {'page_number': meta['page_number'], 'image_path': meta['image_path'], 'similarity': 1 - dist}
        for meta, dist in zip(results['metadatas'][0], results['distances'][0])
    ]

def build_multimodal_input(query_text, pages):
    content = [{'type': 'text', 'text': f'Question: {query_text}'}]
    for p in pages:
        with open(p['image_path'], 'rb') as f:
            image_bytes = f.read()
        content.append({
            'type': 'image',
            'data': base64.b64encode(image_bytes).decode('utf-8'),
            'mime_type': 'image/png',
        })
    return content

def generate_answer(gemini_client, query_text, pages):
    try:
        interaction = gemini_client.interactions.create(
            model=GEMINI_MODEL,
            input=build_multimodal_input(query_text, pages),
            system_instruction=SYSTEM_INSTRUCTION,
        )
        return interaction.output_text
    except Exception as e:
        return f"Gemini request failed: {e}"

# ── Sidebar ──────────────────────────────────────────────────────────────────
st.sidebar.title("Settings")
api_key = st.sidebar.text_input(
    "Gemini API key", type="password",
    value=os.environ.get("GEMINI_API_KEY", ""),
    help="Get one at aistudio.google.com/apikey",
)
top_k = st.sidebar.slider("Pages to retrieve (k)", min_value=1, max_value=5, value=3)
st.sidebar.caption(f"Index: {collection.count()} pages · Device: {DEVICE}")

# ── Main UI ──────────────────────────────────────────────────────────────────
st.title("Visual-Doc Assistant")
st.caption("Multimodal RAG over technical documents — ColPali retrieval + Gemini generation, no OCR.")

query = st.text_input("Ask a question about the document:", placeholder="e.g. What is the system architecture?")
ask_clicked = st.button("Ask", type="primary")

if ask_clicked:
    if not api_key:
        st.error("Please enter your Gemini API key in the sidebar.")
    elif not query.strip():
        st.warning("Please enter a question.")
    else:
        gemini_client = genai.Client(api_key=api_key)
        with st.spinner("Retrieving relevant pages..."):
            pages = retrieve_top_k_pages(query, k=top_k)
        with st.spinner("Generating answer with Gemini..."):
            answer = generate_answer(gemini_client, query, pages)

        st.subheader("Answer")
        st.write(answer)

        st.subheader("Source pages")
        cols = st.columns(len(pages))
        for col, p in zip(cols, pages):
            with col:
                img = Image.open(p['image_path']).convert('RGB')
                caption = f"Page {p['page_number']} · similarity {p['similarity']:.3f}"
                st.image(img, caption=caption, use_container_width=True)

In [ ]:
!cp /content/app.py ./app.py

In [ ]:
import os
path = './app.py'
print("Exists:", os.path.exists(path))
if os.path.exists(path):
    content = open(path).read()
    print("Has device_map fix:", "device_map=DEVICE" in content)
    print("Has custom_text_proj patch:", "_patch_custom_text_proj" in content)

## Step 10 — Launch the App from Colab

Colab can't expose `localhost` directly, so `cloudflare` forwards port 8501 to a public URL. This runs the Streamlit process in the background and prints a tunnel link — open it, and when prompted for a "Tunnel Password," paste the IP printed below.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin/cloudflared

!pip install -q streamlit

!pkill -9 -f streamlit || true
!pkill -9 -f cloudflared || true

import time

!streamlit run ./app.py &>/content/logs.txt &
time.sleep(6)

!cloudflared tunnel --url http://localhost:8501 &>/content/cloudflared_logs.txt &
time.sleep(8)

!grep -o 'https://[a-zA-Z0-9.-]*\.trycloudflare\.com' /content/cloudflared_logs.txt | head -1

## Step 11 — Fully Interactive In-Notebook UI (ipywidgets)

Steps 9–10 stand up the full Streamlit app behind a Cloudflare tunnel — the right
answer for a shareable, deployed demo. This step adds a second, lighter-weight UI
that runs **directly inside this notebook** using `ipywidgets`, so the pipeline can
be demoed live (e.g. in front of the adviser) without depending on a tunnel URL,
a second process, or the network round-trip to `*.trycloudflare.com` staying up.

It does not duplicate any logic — it calls the exact same `retrieve_top_k_pages()`
and `generate_answer()` functions defined in Steps 4–7 above, so results here are
guaranteed identical to the deployed app. Two tabs:

- **💬 Ask** — a text box + Top-k slider + Ask button. Each turn renders the
  retrieved page thumbnails (with cosine similarity) and Gemini's answer inline,
  and keeps a running transcript for the session (with an optional save-to-Drive
  button for including in a report/appendix).
- **🗂 Indexed Pages** — an on-demand thumbnail gallery of every page currently in
  the ChromaDB collection, useful for visually confirming what's actually indexed
  before asking questions about it.

> Run Steps 1–7 first — this cell reuses `model`, `processor`, `collection`, and
> `gemini_client` from those cells rather than reloading anything.

In [ ]:
# Ensure ipywidgets is available and (if running in Colab) enable its widget manager.
!pip install -q ipywidgets

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except ImportError:
    pass  # not running in Colab — ipywidgets renders natively in classic Jupyter/JupyterLab

import io
import time
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

print("ipywidgets ready.")


In [ ]:
# ── Guard: make sure the Phase 3 pipeline (Steps 1-7) has actually been run ───
_required = ["model", "processor", "collection", "gemini_client",
             "retrieve_top_k_pages", "generate_answer"]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing: {', '.join(_missing)}. Run Steps 1-7 above before this cell — "
        "the interactive UI reuses those exact objects/functions so results match "
        "the deployed Streamlit app."
    )

# ── Session state ──────────────────────────────────────────────────────────
_session_history = []  # [{"query", "pages", "answer", "elapsed"}, ...] for this run

# ── "Ask" tab widgets ──────────────────────────────────────────────────────
query_box = widgets.Text(
    placeholder="e.g. What is the system architecture and pipeline diagram?",
    description="Question:",
    style={"description_width": "75px"},
    layout=widgets.Layout(width="75%"),
)
k_slider = widgets.IntSlider(
    value=3, min=1, max=5, step=1, description="Top-k:",
    style={"description_width": "55px"},
    layout=widgets.Layout(width="260px"),
)
ask_button = widgets.Button(
    description="Ask", button_style="primary", icon="search",
    layout=widgets.Layout(width="110px"),
)
clear_button = widgets.Button(
    description="Clear", icon="trash", layout=widgets.Layout(width="110px"),
)
save_button = widgets.Button(
    description="Save transcript", icon="save", layout=widgets.Layout(width="160px"),
)
status_label = widgets.HTML(value="")

qa_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd", padding="12px", margin="10px 0 0 0",
        max_height="560px", overflow="auto",
    )
)


def _page_thumb_html(p):
    img = Image.open(p["image_path"]).convert("RGB")
    img.thumbnail((240, 340))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return (
        "<div style='text-align:center;margin:4px'>"
        f"<img src='data:image/png;base64,{b64}' "
        "style='border:1px solid #ccc;border-radius:6px;"
        "box-shadow:0 1px 3px rgba(0,0,0,.15)'/>"
        f"<div style='font-size:11px;color:#666;margin-top:2px'>"
        f"page {p['page_number']} &middot; sim {p['similarity']:.3f}</div>"
        "</div>"
    )


def _render_turn(query_text, pages, answer, elapsed):
    display(Markdown(f"---\n**Q{len(_session_history)}.** {query_text}"))
    display(widgets.HTML(
        "<div style='display:flex;gap:8px;flex-wrap:wrap;margin:6px 0'>"
        + "".join(_page_thumb_html(p) for p in pages) + "</div>"
    ))
    display(Markdown(f"**Answer** *(retrieved in {elapsed:.1f}s)*:\n\n{answer}"))


def _on_ask_clicked(_btn):
    q = query_box.value.strip()
    if not q:
        status_label.value = "<span style='color:#c00'>Type a question first.</span>"
        return
    ask_button.disabled = True
    status_label.value = "<span style='color:#888'>retrieving pages + calling Gemini...</span>"
    t0 = time.time()
    try:
        pages = retrieve_top_k_pages(q, k=k_slider.value)
        answer = generate_answer(q, pages)
        elapsed = time.time() - t0
        _session_history.append(
            {"query": q, "pages": pages, "answer": answer, "elapsed": elapsed}
        )
        with qa_output:
            _render_turn(q, pages, answer, elapsed)
        status_label.value = f"<span style='color:#080'>done in {elapsed:.1f}s</span>"
        query_box.value = ""
    except Exception as e:
        with qa_output:
            display(Markdown(f"---\n**Error:** `{e}`"))
        status_label.value = "<span style='color:#c00'>request failed — see log</span>"
    finally:
        ask_button.disabled = False


def _on_clear_clicked(_btn):
    _session_history.clear()
    with qa_output:
        clear_output()
    status_label.value = ""


def _on_save_clicked(_btn):
    if not _session_history:
        status_label.value = "<span style='color:#c00'>Nothing to save yet.</span>"
        return
    out_path = "./demo_transcript.md"
    lines = ["# Visual-Doc Assistant — Demo Transcript\n"]
    for i, turn in enumerate(_session_history, 1):
        pages_str = ", ".join(
            f"p.{p['page_number']} ({p['similarity']:.3f})" for p in turn["pages"]
        )
        lines.append(f"## Q{i}. {turn['query']}\n")
        lines.append(f"*Retrieved: {pages_str} &middot; {turn['elapsed']:.1f}s*\n")
        lines.append(f"\n{turn['answer']}\n\n")
    try:
        with open(out_path, "w") as f:
            f.write("\n".join(lines))
        status_label.value = f"<span style='color:#080'>saved to {out_path}</span>"
    except Exception as e:
        status_label.value = f"<span style='color:#c00'>save failed: {e}</span>"


ask_button.on_click(_on_ask_clicked)
clear_button.on_click(_on_clear_clicked)
save_button.on_click(_on_save_clicked)
try:
    query_box.on_submit(_on_ask_clicked)  # Enter key = Ask, on ipywidgets versions that support it
except Exception:
    pass

ask_tab = widgets.VBox([
    widgets.HTML(
        "<p style='color:#666;margin:4px 0'>Reuses <code>retrieve_top_k_pages()</code> and "
        "<code>generate_answer()</code> from Steps 4-7 above — same pipeline as the deployed app.</p>"
    ),
    widgets.HBox([query_box, k_slider]),
    widgets.HBox([ask_button, clear_button, save_button, status_label]),
    qa_output,
])

# ── "Indexed Pages" tab ─────────────────────────────────────────────────────
gallery_output = widgets.Output()
gallery_button = widgets.Button(
    description="Load indexed pages", icon="refresh", button_style="info",
)


def _on_gallery_clicked(_btn):
    with gallery_output:
        clear_output()
        display(widgets.HTML(
            f"<p style='color:#666'>{collection.count()} page(s) indexed in ChromaDB</p>"
        ))
        try:
            data = collection.get(include=["metadatas"])
            metas = sorted(data["metadatas"], key=lambda m: m["page_number"])
            thumbs = []
            for m in metas:
                img = Image.open(m["image_path"]).convert("RGB")
                img.thumbnail((150, 210))
                buf = io.BytesIO()
                img.save(buf, format="PNG")
                b64 = base64.b64encode(buf.getvalue()).decode()
                thumbs.append(
                    "<div style='text-align:center;margin:4px'>"
                    f"<img src='data:image/png;base64,{b64}' "
                    "style='border:1px solid #ccc;border-radius:4px'/>"
                    f"<div style='font-size:11px;color:#666'>page {m['page_number']}</div></div>"
                )
            display(widgets.HTML(
                "<div style='display:flex;flex-wrap:wrap;max-height:520px;overflow:auto'>"
                + "".join(thumbs) + "</div>"
            ))
        except Exception as e:
            display(Markdown(f"**Could not load gallery:** `{e}`"))


gallery_button.on_click(_on_gallery_clicked)
pages_tab = widgets.VBox([gallery_button, gallery_output])

# ── Assemble & display ──────────────────────────────────────────────────────
tabs = widgets.Tab(children=[ask_tab, pages_tab])
tabs.set_title(0, "Ask")
tabs.set_title(1, "Indexed Pages")

display(widgets.HTML(
    "<h3 style='margin-bottom:2px'>Visual-Doc Assistant — Interactive Demo</h3>"
    "<p style='color:#666;margin-top:0'>Live, in-notebook UI — ask questions and see "
    "retrieval + Gemini's answer inline, no Streamlit/Cloudflare tunnel required.</p>"
))
display(tabs)


---
## Phase 3 Summary

| Component | Detail |
|---|---|
| Retrieval | `embed_query()` (from Phase 2) → ChromaDB cosine search → Top-K pages |
| Generation model | Gemini 3.6 Flash (`gemini-3.6-flash`) via the `google-genai` Interactions API |
| Prompt construction | Query text + inline base64 page image(s) (PNG) + a grounding system instruction |
| Answer grounding | System instruction restricts Gemini to the retrieved page images only |
| UI | Streamlit app (`app.py`) — query box, Ask button, answer + source-page thumbnails, **plus** an in-notebook `ipywidgets` UI (Step 11) for tunnel-free live demos |
| Deployment | Runs inside Colab (GPU needed for ColPali), exposed via cloudflare |

### What's not in scope here
- **Uploading a new PDF through the UI.** Phases 1–2 indexed one document (the ColPali paper, 26 pages). Supporting a fresh upload would mean wiring Phase 1's PDF→image step and Phase 2's embedding loop into the Streamlit app itself.
- **Full late-interaction (MaxSim) retrieval.** Still mean-pooled to a single 128-dim vector per page, as noted in Phase 2. PyColBERT-style late interaction remains a documented upgrade path, not implemented here.

---
*Visual-Doc Assistant — NIT Patna, MCA (DS&I), Course: MC460502*